# MMSI language analysis: blind Qwen2.5-VL-7B

Scores every question **without images**, using next-token logits for A, B, C, and D.
Conditions: original prompt, options only, and three cyclic option rotations.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Upload `mmsi_blind_vlm_colab.zip` when asked. It contains the code and the text cache only.
This notebook writes a new `outputs/05_colab_raw.jsonl`. Do not upload the Mac smoke-test JSONL; those rows were scored with disk offloading.

The first scoring line must say `Loaded Qwen/Qwen2.5-VL-7B-Instruct (4bit)`.
Stop the run if it says parameters were offloaded to disk.

In [ ]:
import shutil
import subprocess
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError('No GPU visible. Use Runtime -> Change runtime type -> T4 GPU, then rerun this cell.')

In [ ]:
import os
import zipfile
from pathlib import Path
from google.colab import files

ROOT = Path('/content/mmsi_language')
if ROOT.exists():
    shutil.rmtree(ROOT)
print('Choose mmsi_blind_vlm_colab.zip')
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content')
os.chdir(ROOT)
required = [
    ROOT / 'common.py',
    ROOT / 'blind_vlm.py',
    ROOT / '05_blind_vlm.py',
    ROOT / 'cache' / 'mmsi_text.parquet',
    ROOT / 'outputs' / '01_tags.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Bundle is missing: ' + ', '.join(missing))
stale = list((ROOT / 'outputs').glob('05_*.jsonl')) + list((ROOT / 'outputs').glob('05_*.csv'))
if stale:
    raise RuntimeError('Bundle contains previous Step E outputs. Rebuild the zip so Colab starts fresh.')
print('Unpacked', ROOT)

In [ ]:
%pip install -q -r requirements-blind-vlm.txt

## Smoke test

Eight questions, 40 prompts. This checks that 4-bit loading works before the 5,000-prompt run.
If this cell asks you to restart the runtime, use Runtime → Restart session, then rerun from the GPU-check cell.

In [ ]:
!python 05_blind_vlm.py --limit 8 --quantization 4bit --batch-size 4 --raw-path outputs/05_colab_raw.jsonl

## Full run

About 5,000 prompts. The eight smoke-test prompts are skipped because they are already in the same raw file.
If Colab disconnects, upload the partial `05_colab_raw.jsonl` into `outputs/` and rerun this cell; completed keys are resumed.
Expect roughly 1–2 hours on a T4. Reduce `--batch-size` to 2 if CUDA runs out of memory.

In [ ]:
!python 05_blind_vlm.py --quantization 4bit --batch-size 4 --raw-path outputs/05_colab_raw.jsonl | tee outputs/05_run.log

In [ ]:
import pandas as pd
summary = pd.read_csv('outputs/05_blind_vlm_summary.csv')
display(summary[summary['metric'].str.contains('accuracy|correct|consistent|max_input')].round(4))

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/mmsi_blind_vlm_results', 'zip', 'outputs')
files.download('/content/mmsi_blind_vlm_results.zip')